<a href="https://colab.research.google.com/github/07Sathwika07/EDAV-PROJECT/blob/main/Online_Advertisement_Click_Through_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Online Advertisement Click-Through Analysis
**College-level Python Data Analytics Project**

## Project objective
Explore an online advertising dataset to understand which user, browsing, and advertisement attributes are associated with an ad click. The project focuses on reproducible exploratory data analysis (EDA), clear visualizations, data-quality checks, and practical business recommendations.

> **Dataset note:** No dataset was attached when this project pack was created. This notebook is designed to inspect an uploaded CSV and adapt to its columns. A commonly used practice dataset is the Kaggle *Advertising* dataset, often containing fields such as `Daily Time Spent on Site`, `Age`, `Area Income`, `Daily Internet Usage`, `Ad Topic Line`, `City`, `Male`, `Timestamp`, and `Clicked on Ad`. Verify the dataset's license and provenance on its download page before reuse. The notebook does not assume those columns must exist.

## Step 1 — Workflow
1. Load a CSV file (upload in Colab or place it beside this notebook).
2. Inspect shape, columns, data types, missing values, duplicates, and sample rows.
3. Detect a likely click/target column and normalize common target labels where possible.
4. Clean column names, duplicate rows, text whitespace, numeric values, and date fields conservatively.
5. Explore the target distribution and relationships with available numeric/categorical/date features.
6. Generate at least six visualizations when the dataset has enough relevant columns.
7. Summarize calculated findings and translate associations into cautious business recommendations.

The notebook reports actual values from the dataset you load. It does not contain fabricated results.

## Step 2 — Import libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 100)

## Step 3 — Load the dataset

In [ ]:
# OPTION A (Google Colab): run this cell and select your CSV file.
try:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if csv_files:
        DATA_PATH = csv_files[0]
    else:
        DATA_PATH = None
except ImportError:
    # OPTION B (Jupyter / VS Code): place the CSV next to this notebook
    # and change the filename below if needed.
    DATA_PATH = "advertising.csv"

if DATA_PATH and Path(DATA_PATH).exists():
    df_raw = pd.read_csv(DATA_PATH)
    print(f"Loaded: {DATA_PATH}")
    print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
    display(df_raw.head())
else:
    df_raw = None
    print("Dataset not found. Upload a CSV in Colab or set DATA_PATH to your local CSV file.")

## Step 4 — Inspect the dataset

In [3]:
if df_raw is not None:
    print("Column names:")
    print(df_raw.columns.tolist())
    print("\nData types:")
    display(df_raw.dtypes.rename("dtype").to_frame())
    print("\nDescriptive statistics:")
    display(df_raw.describe(include="all").T)
    print("\nMissing values:")
    missing = df_raw.isna().sum().sort_values(ascending=False)
    display(pd.DataFrame({
        "missing_count": missing,
        "missing_percent": (missing / len(df_raw) * 100).round(2)
    }))
    print("\nExact duplicate rows:", int(df_raw.duplicated().sum()))

## Step 5 — Standardize names and clean conservatively
Column names are converted to lowercase snake_case to make them easier to reference. Exact duplicate rows are removed, because identical full records are usually repeated exports; if repeated rows are valid repeated events in your business context, revisit this decision. Missing values are not blindly filled: the EDA below uses available observations and reports missingness.

In [4]:
if df_raw is not None:
    df = df_raw.copy()
    df.columns = (
        df.columns.astype(str).str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    before = len(df)
    df = df.drop_duplicates().copy()
    print(f"Removed {before - len(df):,} exact duplicate rows.")

    # Trim whitespace in text columns; preserve genuine missing values.
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

    print("Cleaned shape:", df.shape)
    display(df.head())

## Step 6 — Identify a likely click target

In [5]:
def find_column(columns, candidates):
    # Exact normalized match first, then substring match.
    cols = list(columns)
    for candidate in candidates:
        if candidate in cols:
            return candidate
    for col in cols:
        if any(candidate in col for candidate in candidates):
            return col
    return None

if df_raw is not None:
    target_col = find_column(df.columns, [
        "clicked_on_ad", "clicked", "click", "is_clicked", "click_through",
        "clickthrough", "converted", "conversion", "target", "label"
    ])
    print("Likely target column:", target_col)
    if target_col:
        print("Original target values:")
        display(df[target_col].value_counts(dropna=False).to_frame("count"))

### Target interpretation
The notebook can analyze target values as categories even when their labels are unusual. If your dataset uses values such as `0/1`, `Yes/No`, or `True/False`, the helper below attempts to map them to `No click` and `Click`. Confirm that the chosen target really means an ad click; a column called `conversion` may represent a later action instead.

In [6]:
if df_raw is not None and target_col:
    def normalize_click_target(series):
        s = series.astype("string").str.strip().str.lower()
        mapping = {
            "1": "Click", "1.0": "Click", "true": "Click", "yes": "Click",
            "y": "Click", "clicked": "Click", "click": "Click",
            "0": "No click", "0.0": "No click", "false": "No click", "no": "No click",
            "n": "No click", "not clicked": "No click", "no click": "No click"
        }
        mapped = s.map(mapping)
        # Keep original labels if they are not a recognized binary encoding.
        df["click_outcome"] = mapped.fillna(series.astype("string").str.strip())
        display(df["click_outcome"].value_counts(dropna=False).to_frame("count"))

## Step 7 — Validate numeric and date fields

In [7]:
if df_raw is not None:
    # Try parsing columns whose names suggest dates/timestamps.
    date_cols = [c for c in df.columns if any(k in c for k in ["date", "time", "timestamp"])]
    parsed_dates = {}
    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors="coerce")
        # Keep only if a reasonable share of non-missing values parses.
        non_missing = df[col].notna().sum()
        success_rate = (parsed.notna().sum() / non_missing) if non_missing else 0
        if success_rate >= 0.5:
            parsed_dates[col] = parsed
            df[col + "_parsed"] = parsed
            print(f"{col}: parsed {success_rate:.1%} of non-missing values")

    # Convert object columns to numeric only when most non-missing entries convert.
    for col in df.select_dtypes(include=["object", "string"]).columns:
        if col in ["click_outcome"]:
            continue
        converted = pd.to_numeric(df[col], errors="coerce")
        non_missing = df[col].notna().sum()
        if non_missing and converted.notna().sum() / non_missing >= 0.9:
            df[col] = converted

    print("Updated data types:")
    display(df.dtypes.rename("dtype").to_frame())

## Step 8 — Summary metrics

In [8]:
if df_raw is not None:
    print(f"Rows available for analysis: {len(df):,}")
    print(f"Columns available for analysis: {df.shape[1]}")
    print(f"Missing cells remaining: {int(df.isna().sum().sum()):,}")
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
    print("Numeric columns:", numeric_cols)
    print("Categorical columns:", categorical_cols)
    if "click_outcome" in df.columns:
        print("\nClick outcome distribution:")
        display(df["click_outcome"].value_counts(normalize=True).mul(100).round(2).rename("percent").to_frame())

## Step 9 — Visualizations

The following cells create charts only when suitable columns exist. For the commonly used advertising dataset, the target distribution, numeric feature distributions, and feature-vs-click charts are usually available.

In [9]:
if df_raw is not None:
    # Visualization 1: target / click outcome distribution
    if "click_outcome" in df.columns:
        plt.figure()
        order = df["click_outcome"].value_counts().index
        sns.countplot(data=df, x="click_outcome", order=order)
        plt.title("Distribution of Advertisement Click Outcomes")
        plt.xlabel("Click outcome")
        plt.ylabel("Number of records")
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()
    else:
        print("Visualization 1 skipped: no likely click target was detected.")

In [10]:
if df_raw is not None:
    numeric_cols = [c for c in df.select_dtypes(include=np.number).columns
                    if df[c].nunique(dropna=True) > 1]
    # Visualization 2: distributions of up to four numeric features
    if numeric_cols:
        selected = numeric_cols[:4]
        df[selected].hist(bins=25, figsize=(12, 8), edgecolor="black")
        plt.suptitle("Distributions of Numeric Features", y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print("Visualization 2 skipped: no usable numeric features.")

In [11]:
if df_raw is not None:
    # Visualization 3: correlation heatmap
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(10, 7))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
        plt.title("Correlation Heatmap of Numeric Features")
        plt.tight_layout()
        plt.show()
    else:
        print("Visualization 3 skipped: fewer than two numeric columns.")

In [12]:
if df_raw is not None:
    # Visualization 4: numeric features by click outcome
    if "click_outcome" in df.columns and numeric_cols:
        feature = next((c for c in numeric_cols if df[c].nunique(dropna=True) > 2), None)
        if feature:
            plt.figure(figsize=(9, 5))
            sns.boxplot(data=df, x="click_outcome", y=feature)
            plt.title(f"{feature.replace('_', ' ').title()} by Click Outcome")
            plt.xlabel("Click outcome")
            plt.ylabel(feature.replace("_", " ").title())
            plt.xticks(rotation=15)
            plt.tight_layout()
            plt.show()
        else:
            print("Visualization 4 skipped: no numeric feature with enough distinct values.")
    else:
        print("Visualization 4 skipped: target or numeric features unavailable.")

In [13]:
if df_raw is not None:
    # Visualization 5: click rate by a categorical feature with manageable cardinality
    cat_candidates = [c for c in df.select_dtypes(include=["object", "string", "category", "bool"]).columns
                      if c not in ["click_outcome", target_col] and 2 <= df[c].nunique(dropna=True) <= 15]
    if "click_outcome" in df.columns and cat_candidates:
        cat = cat_candidates[0]
        temp = df[[cat, "click_outcome"]].dropna().copy()
        temp["is_click"] = temp["click_outcome"].astype(str).str.lower().isin(["click", "clicked", "1", "true", "yes"]).astype(int)
        rate = temp.groupby(cat, observed=False)["is_click"].mean().sort_values(ascending=False)
        plt.figure(figsize=(10, 5))
        sns.barplot(x=rate.index.astype(str), y=rate.values)
        plt.title(f"Observed Click Rate by {cat.replace('_', ' ').title()}")
        plt.xlabel(cat.replace("_", " ").title())
        plt.ylabel("Observed click rate")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Visualization 5 skipped: no suitable low-cardinality category and click target.")

In [14]:
if df_raw is not None:
    # Visualization 6: pairplot on a small subset, if possible
    numeric_cols = [c for c in df.select_dtypes(include=np.number).columns if df[c].nunique(dropna=True) > 1]
    if len(numeric_cols) >= 2:
        pair_cols = numeric_cols[:3]
        pair_data = df[pair_cols + (["click_outcome"] if "click_outcome" in df.columns else [])].dropna()
        if len(pair_data) > 0:
            sns.pairplot(pair_data.sample(min(len(pair_data), 500), random_state=42),
                         hue="click_outcome" if "click_outcome" in pair_data.columns else None,
                         diag_kind="hist")
            plt.suptitle("Pairwise Relationships Between Selected Features", y=1.02)
            plt.show()
    else:
        print("Visualization 6 skipped: fewer than two usable numeric features.")

In [15]:
if df_raw is not None:
    # Visualization 7: monthly trend, if a date column was parsed
    parsed_date_cols = [c for c in df.columns if c.endswith("_parsed") and df[c].notna().any()]
    if parsed_date_cols:
        date_col = parsed_date_cols[0]
        trend = df.dropna(subset=[date_col]).copy()
        trend["month"] = trend[date_col].dt.to_period("M").astype(str)
        monthly = trend.groupby("month").size()
        plt.figure(figsize=(11, 5))
        monthly.plot(marker="o")
        plt.title(f"Record Volume Over Time ({date_col.replace('_parsed','')})")
        plt.xlabel("Month")
        plt.ylabel("Number of records")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Visualization 7 skipped: no date/time column parsed successfully.")

## Step 10 — Calculated findings (no invented conclusions)

In [16]:
if df_raw is not None:
    print("DATASET SUMMARY")
    print("-" * 60)
    print("Records after exact-duplicate removal:", len(df))
    print("Features:", df.shape[1])
    print("Missing cells:", int(df.isna().sum().sum()))
    if "click_outcome" in df.columns:
        counts = df["click_outcome"].value_counts(dropna=False)
        print("\nOutcome counts:")
        print(counts.to_string())
        known = df["click_outcome"].dropna().astype(str).str.lower()
        click_labels = {"click", "clicked", "1", "1.0", "true", "yes", "y"}
        click_rate = known.isin(click_labels).mean() if len(known) else np.nan
        print(f"Click rate among recognized click/no-click labels: {click_rate:.2%}" if pd.notna(click_rate)
              else "Click rate not calculated: labels could not be mapped confidently.")
    print("\nNumeric summary:")
    display(df.select_dtypes(include=np.number).describe().T)

## Step 11 — Interpretation guide
Use the output to write findings specific to your data:
- **Click rate:** proportion of records labelled as clicks, only if target labels are understood.
- **Feature associations:** compare distributions and observed click rates across groups. A relationship is not proof that a feature causes clicks.
- **Correlation:** measures linear association between numeric columns; it does not prove causation and may miss nonlinear patterns.
- **Data quality:** missing values, duplicates, extreme values, class imbalance, and ambiguous target definitions can distort conclusions.
- **Privacy and ethics:** do not use personal attributes to make unfair or sensitive targeting decisions. Follow the dataset license and applicable privacy rules.

## Step 12 — Optional machine-learning extension
A predictive model is not required for an EDA project. If you extend it, first confirm the target, remove leakage columns, split into train/test sets, use a preprocessing pipeline for missing values and categorical variables, and report precision, recall, F1, ROC-AUC (where appropriate), and a baseline. Do not report model performance until it has actually been trained and evaluated on the dataset.

## Conclusion template
After running all cells, summarize (1) the actual observed click rate, (2) the strongest descriptive patterns, (3) data-quality limitations, and (4) 2–4 practical experiments a marketing team could test. Replace this template with measured results; do not write conclusions before executing the notebook.